In [4]:
"""
LSL EEG Recorder – Happy condition
Records EEG data from an OpenBCI LSL stream while playing YouTube stimuli
defined in a Google Sheet, saving one CSV per video clip.
"""

import io
import os
import threading
import time
import urllib.parse
from pathlib import Path

import pandas as pd
import requests
from pylsl import StreamInlet, resolve_byprop, local_clock
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# ── Configuration ────────────────────────────────────────────────────────────

SHEET_URL = (
    "https://docs.google.com/spreadsheets/d/"
    "1xvwLRa_QE4QsopQWWJBoMO-lzheScZMt5OZLdEFLDng"
    "/export?format=csv&gid=0"
)
LSL_STREAM_NAME    = "obci_eeg1"
CHROME_DEBUGGER    = "localhost:9222"
OUT_FOLDER         = Path("recordings_sad")
CROSSHAIR_DURATION = 10          # seconds the fixation cross is shown

import urllib.parse

_CROSSHAIR_HTML = """<!DOCTYPE html>
<html><head><style>
* { margin:0; padding:0; box-sizing:border-box; }
body { background:#000; width:100vw; height:100vh;
       display:flex; align-items:center; justify-content:center; }
.h { position:absolute; width:60px; height:4px; background:#00c800;
     top:50%; left:50%; transform:translate(-50%,-50%); }
.v { position:absolute; width:4px; height:60px; background:#00c800;
     top:50%; left:50%; transform:translate(-50%,-50%); }
</style></head>
<body><div class="h"></div><div class="v"></div></body>
</html>"""

_CROSSHAIR_URI = "data:text/html;charset=utf-8," + urllib.parse.quote(_CROSSHAIR_HTML)

# Column indices inside the Google Sheet
COL_NAME  = 2
COL_URL   = 4
COL_START = 5
COL_END   = 6

# ── Helpers ───────────────────────────────────────────────────────────────────

def parse_time_to_seconds(value) -> int:
    """Convert a spreadsheet time cell (MM:SS or raw seconds) to an int."""
    if pd.isna(value):
        return 0
    text = str(value).strip()
    if ":" in text:
        try:
            m, s = map(int, text.split(":"))
            return m * 60 + s
        except ValueError:
            return 0
    try:
        return int(float(text))
    except ValueError:
        return 0


def extract_video_id(raw_url: str) -> str:
    """Return the YouTube video ID from a full URL or bare ID string."""
    if "v=" in raw_url:
        return raw_url.split("v=")[-1].split("&")[0]
    return raw_url.split("/")[-1].split("?")[0]


def safe_filename(name: str) -> str:
    """Strip unsafe characters and return an underscore-delimited filename stem."""
    cleaned = "".join(c for c in name if c.isalnum() or c in (" ", "_")).strip()
    return cleaned.replace(" ", "_")


def load_sheet(url: str) -> pd.DataFrame:
    """Download and parse the stimulus Google Sheet."""
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
    response.raise_for_status()
    return pd.read_csv(io.StringIO(response.content.decode("utf-8-sig")), encoding="utf-8")

# ── LSL connection ────────────────────────────────────────────────────────────

def connect_lsl(stream_name: str) -> StreamInlet:
    """Resolve and return an LSL StreamInlet, raising if not found."""
    print(f"Searching for LSL stream '{stream_name}' …")
    streams = resolve_byprop("name", stream_name, timeout=10)
    if not streams:
        raise RuntimeError(f"LSL stream '{stream_name}' not found.")
    inlet = StreamInlet(streams[0])
    print("✓ LSL connected")
    return inlet


def get_channel_labels(inlet: StreamInlet) -> list[str]:
    count = inlet.channel_count if isinstance(inlet.channel_count, int) else inlet.channel_count()
    return [f"EEG_{i + 1}" for i in range(count)]

# ── Selenium helpers ──────────────────────────────────────────────────────────

def attach_chrome(debugger_address: str) -> webdriver.Chrome:
    """Attach Selenium to an already-running Chrome instance."""
    opts = Options()
    opts.add_experimental_option("debuggerAddress", debugger_address)
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts,
    )


def show_crosshair(driver: webdriver.Chrome, duration: int = CROSSHAIR_DURATION) -> None:
    """Navigate to a data URI page showing a green fixation cross for `duration` seconds."""
    driver.get(_CROSSHAIR_URI)
    print(f"  ✛ Fixation cross ({duration}s) …", end=" ", flush=True)
    time.sleep(duration)
    print("done")


def play_video(driver: webdriver.Chrome, timeout: int = 20) -> None:
    """
    Wait for a <video> element, then attempt to start playback.
    Raises RuntimeError if the video never starts.
    """
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.TAG_NAME, "video"))
    )
    for _ in range(10):
        try:
            video = driver.find_element(By.TAG_NAME, "video")
            driver.execute_script("arguments[0].play();", video)
            if not driver.execute_script("return arguments[0].paused;", video):
                return          # playing ✓
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError("Video did not start within the allotted time.")

# ── Recording ─────────────────────────────────────────────────────────────────

def record_loop(
    inlet: StreamInlet,
    time_offset: float,
    start_time: float,
    video_name: str,
    buffer: list,
    stop_event: threading.Event,
) -> None:
    """Pull LSL samples until stop_event is set, appending rows to buffer."""
    while not stop_event.is_set():
        sample, ts = inlet.pull_sample(timeout=0.1)
        if sample:
            corrected_ts  = ts + time_offset
            relative_time = corrected_ts - start_time
            buffer.append([relative_time] + sample + [video_name])


def save_recording(
    data: list,
    channel_labels: list[str],
    trial: int,
    name: str,
    out_folder: Path,
) -> None:
    """Persist a recording buffer to CSV named trial_i_songname.csv."""
    columns  = ["relative_time"] + channel_labels + ["video"]
    filename = out_folder / f"trial_{trial}_{safe_filename(name)}.csv"
    pd.DataFrame(data, columns=columns).to_csv(filename, index=False)
    print(f"  ✓ Saved: {filename}")

# ── Main orchestration ────────────────────────────────────────────────────────

def run() -> None:
    df     = load_sheet(SHEET_URL)
    inlet  = connect_lsl(LSL_STREAM_NAME)
    offset = inlet.time_correction()
    labels = get_channel_labels(inlet)

    driver = attach_chrome(CHROME_DEBUGGER)
    OUT_FOLDER.mkdir(parents=True, exist_ok=True)

    for trial, (_, row) in enumerate(df.iterrows(), start=1):
        name    = str(row.iloc[COL_NAME])
        raw_url = str(row.iloc[COL_URL])
        start_s = parse_time_to_seconds(row.iloc[COL_START])
        end_s   = parse_time_to_seconds(row.iloc[COL_END])
        duration = max(0, end_s - start_s)

        v_id = extract_video_id(raw_url)
        url  = f"https://www.youtube.com/watch?v={v_id}&t={start_s}s"

        print(f"\n▶  Trial {trial}: {name}  ({start_s}–{end_s}s)")

        try:
            show_crosshair(driver)

            driver.get(url)
            play_video(driver)

            # Flush stale samples buffered during crosshair / page-load pause
            while True:
                sample, _ = inlet.pull_sample(timeout=0.0)
                if not sample:
                    break

            buffer     = []
            stop_event = threading.Event()
            start_time = local_clock() + offset

            thread = threading.Thread(
                target=record_loop,
                args=(inlet, offset, start_time, name, buffer, stop_event),
                daemon=True,
            )
            thread.start()
            time.sleep(duration)
            stop_event.set()
            thread.join()

            if buffer:
                save_recording(buffer, labels, trial, name, OUT_FOLDER)
            else:
                print("  ⚠ No samples recorded.")

        except Exception as exc:
            print(f"  ⚠ Skipping '{name}': {exc}")

        time.sleep(2)   # brief pause between stimuli

    driver.quit()
    print("\n✓ Session complete.")


if __name__ == "__main__":
    try:
        run()
    except Exception as exc:
        print(f"Fatal error: {exc}")

Searching for LSL stream 'obci_eeg1' …
✓ LSL connected

▶  Trial 1: True Love Waits  (60–120s)
  ✛ Fixation cross (10s) … done
  ✓ Saved: recordings_sad\trial_1_True_Love_Waits.csv

▶  Trial 2: Dearly Departed  (60–120s)
  ✛ Fixation cross (10s) … done
  ✓ Saved: recordings_sad\trial_2_Dearly_Departed.csv

▶  Trial 3: Fix You  (60–120s)
  ✛ Fixation cross (10s) … done
  ✓ Saved: recordings_sad\trial_3_Fix_You.csv

▶  Trial 4: O  (60–120s)
  ✛ Fixation cross (10s) … done
  ✓ Saved: recordings_sad\trial_4_O.csv

▶  Trial 5: Happens to the Heart  (60–120s)
  ✛ Fixation cross (10s) … done
  ✓ Saved: recordings_sad\trial_5_Happens_to_the_Heart.csv

▶  Trial 6: Dawn Chorus  (60–120s)
  ✛ Fixation cross (10s) … done
  ✓ Saved: recordings_sad\trial_6_Dawn_Chorus.csv

▶  Trial 7: Nude  (60–120s)
  ✛ Fixation cross (10s) … done
  ✓ Saved: recordings_sad\trial_7_Nude.csv

▶  Trial 8: And I Love Her  (60–120s)
  ✛ Fixation cross (10s) … done
  ✓ Saved: recordings_sad\trial_8_And_I_Love_Her.csv

▶ 